# Feature Engineering & Preprocessing Pipeline

**Goal:** Build a reusable, leak-free preprocessing workflow with missing-value imputation, numerical scaling, categorical one-hot encoding, correlation analysis, and feature importance.

This notebook creates a **synthetic housing-prices-style dataset** so it runs without downloading a dataset. Replace it with a permitted real tabular dataset when ready. All preprocessing is fitted on the training split only through a Scikit-Learn `Pipeline` and `ColumnTransformer`.

## 1. Imports

Required libraries: pandas, numpy, matplotlib, and scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Create a sample tabular dataset

The sample is synthetic and for demonstration only. It is not real housing market data.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_rows = 600

area_sqft = rng.normal(1500, 450, n_rows).clip(450, 4000)
bedrooms = rng.integers(1, 6, n_rows)
age_years = rng.integers(0, 60, n_rows)
neighborhood = rng.choice(['Urban', 'Suburban', 'Rural'], n_rows, p=[0.40, 0.40, 0.20]).astype(object)
property_type = rng.choice(['Apartment', 'Detached', 'Townhouse'], n_rows).astype(object)

neighborhood_effect = pd.Series(neighborhood).map({'Urban': 35000, 'Suburban': 20000, 'Rural': 5000}).to_numpy()
type_effect = pd.Series(property_type).map({'Apartment': 10000, 'Detached': 30000, 'Townhouse': 18000}).to_numpy()
price = 65000 + area_sqft * 145 + bedrooms * 9000 - age_years * 1200 + neighborhood_effect + type_effect + rng.normal(0, 25000, n_rows)

df = pd.DataFrame({
    'area_sqft': area_sqft,
    'bedrooms': bedrooms.astype(float),
    'age_years': age_years.astype(float),
    'neighborhood': neighborhood,
    'property_type': property_type,
    'price': price
})

# Add a small amount of missingness to demonstrate imputation.
for col in ['area_sqft', 'bedrooms', 'age_years', 'neighborhood']:
    missing_idx = rng.choice(df.index, size=int(0.05 * n_rows), replace=False)
    df.loc[missing_idx, col] = np.nan

df.head()

## 3. Inspect data quality and target distribution

Do this before modeling to understand missingness and basic data shape.

In [ ]:
print('Shape:', df.shape)
display(df.isna().sum().rename('missing_values').to_frame())
display(df.describe(include='all').T)

## 4. Correlation analysis

Pearson correlation is calculated for numeric columns only. Correlation indicates association, not causation. Since missing values may be present, pandas uses available pairs.

In [ ]:
numeric_corr = df.select_dtypes(include='number').corr(method='pearson')
display(numeric_corr)

fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(numeric_corr, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(numeric_corr.columns)), numeric_corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(numeric_corr.index)), numeric_corr.index)
ax.set_title('Numeric Feature Correlation Matrix')
fig.colorbar(image, ax=ax, label='Pearson correlation')
plt.tight_layout()
plt.show()

## 5. Split before fitting transformations

The target is `price`. Split first; the imputer, scaler, and encoder will be fitted only on `X_train` inside the pipeline.

In [ ]:
TARGET = 'price'
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include='number').columns.tolist()
categorical_features = X_train.select_dtypes(exclude='number').columns.tolist()

print('Train rows:', len(X_train), '| Test rows:', len(X_test))
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 6. Build reusable preprocessing pipelines

- Numerical: median imputation + standard scaling.
- Categorical: most-frequent imputation + one-hot encoding.
- `handle_unknown='ignore'` allows unseen categories at inference time.
- Target encoding is intentionally not used here; one-hot encoding is a safe, straightforward baseline for this example.

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
])

model_pipeline

## 7. Fit and evaluate

The entire preprocessing pipeline is fitted on training data only. The test set is used only for final evaluation.

In [ ]:
model_pipeline.fit(X_train, y_train)
predictions = model_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f'MAE:  {mae:,.2f}')
print(f'RMSE: {rmse:,.2f}')
print(f'R²:   {r2:.3f}')

## 8. Feature importance using mutual information

Mutual information is calculated on the imputed/scaled/encoded training features only. It is a univariate ranking and does not establish causation.

In [ ]:
fitted_preprocessor = model_pipeline.named_steps['preprocessor']
X_train_transformed = fitted_preprocessor.transform(X_train)
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(X_train_transformed, 'toarray'):
    X_train_transformed = X_train_transformed.toarray()

mi_scores = mutual_info_regression(
    X_train_transformed, y_train, random_state=RANDOM_STATE
)
mi_ranking = pd.Series(mi_scores, index=feature_names, name='mutual_information').sort_values(ascending=False)
display(mi_ranking.to_frame())

mi_ranking.head(15).sort_values().plot(kind='barh', figsize=(8, 5), title='Top Features by Mutual Information')
plt.xlabel('Mutual information score')
plt.tight_layout()
plt.show()

## 9. Reuse the fitted pipeline

Pass a DataFrame with the same input columns to `predict`; the fitted pipeline handles imputation, scaling, encoding, and prediction.

In [ ]:
new_example = pd.DataFrame([{
    'area_sqft': 1600,
    'bedrooms': 3,
    'age_years': 12,
    'neighborhood': 'Suburban',
    'property_type': 'Detached'
}])

print('Example predicted price:', round(float(model_pipeline.predict(new_example)[0]), 2))

## 10. Leakage and responsible-use checklist

- [x] Train/test split occurs before fitting imputers, scalers, and encoders.
- [x] All preprocessing is contained in a Scikit-Learn `Pipeline` / `ColumnTransformer`.
- [x] Test set is held out for final evaluation.
- [x] Example data is synthetic; no personal data is included.
- [ ] If replacing the sample data, verify permission, provenance, target definition, and feature availability at prediction time.
- [ ] For time-dependent data, choose a time-based split instead of a random split when appropriate.
- [ ] Review target leakage, duplicates, subgroup performance, and deployment risks before real use.

## Limitations

This notebook is an educational starter, not a production-ready model. Results from synthetic data should not be interpreted as real housing-price performance. Target encoding is not included; if needed, use a cross-fitting-capable encoder and validate it carefully to avoid leakage.